# County split transformer (prototype)

This notebook calculates the ratio of energy used in each county of Sweden (21). It is a rough prototype without much sophisticated thought.

The data used is energy use data from SCB, downloaded via their API. All energy sources are used.

We average over 10 years to catch at least a few reported numbers from all counties.

### Possible improvements

- Batch the API call so that we can download all data (users and energy sources)
- Do not include renewable energy sources in the calculation as these are less likely to be replaced by electricity
- Look at the trends in energy use for each county
- Find the responsible person at SCB and investiag

In [1]:
# Imports and general variables

import pandas as pd 
import numpy as np
import requests
from io import StringIO

In [6]:
# Load the data from SCB API

url = "https://api.scb.se/OV0104/v1/doris/sv/ssd/START/EN/EN0203/EN0203A/SlutAnvSektor"
query = {
  "query": [
    {
      "code": "Region",
      "selection": {
        "filter": "item",
        "values": ["01","03","04","05","06","07","08","09","10","12","13","14","17","18","19","20","21","22","23","24","25"]
      }
    },
    {
      "code": "Forbrukningskategri",
      "selection": {
        "filter": "item",
        "values": ["999"
        ]
      }
    },
    {
      "code": "Bransle",
      "selection": {
        "filter": "item",
        "values": ["905","910","915","920","925","930","14","16","955"]
      }
    }
  ],
  "response": {
    "format": "csv"
  }
}

##  Make the request (POST)
response = requests.post(url, json=query)
if response.status_code == 200:
    csv_data = StringIO(response.text)
    response_csv = pd.read_csv(csv_data)    
else:
    print(f"Error: {response.status_code}")

## Format dataframe
energy_data = response_csv.copy()
energy_data['Lan'] = energy_data['region'].apply(lambda x: x.split(' ', 1)[0]) # Split the municipality code and the name
energy_data.rename(columns={'Lan': 'geography'}, inplace=True)
energy_data.set_index('geography', inplace=True) # Set the code to index
energy_data = energy_data.drop(columns=['region'])
energy_data.loc[:, ~energy_data.columns.isin(['förbrukarkategori', 'bränsletyp'])] = ( # Make all numbers float
    energy_data.loc[:, ~energy_data.columns.isin(['förbrukarkategori', 'bränsletyp'])]
    .replace("..", np.nan)
    .astype(float)
)
energy_data.rename( # Rename the columns
    columns=lambda col: col.split()[-1] if col.startswith('Slutanvändning (MWh)') else col,
    inplace=True
)

In [8]:
total_energy = energy_data[energy_data['bränsletyp'] == 'totalt'].drop(columns=['förbrukarkategori', 'bränsletyp'])
total_energy['last_three_avg'] = total_energy.loc[:,['2020', '2021', '2022']].mean(axis=1)
total_energy['last_three_avg_ratio'] = total_energy['last_three_avg']/total_energy['last_three_avg'].sum()

total_energy['last_five_avg'] = total_energy.loc[:,['2018', '2019', '2020', '2021', '2022']].mean(axis=1)
total_energy['last_five_avg_ratio'] = total_energy['last_five_avg']/total_energy['last_five_avg'].sum()

total_energy['last_ten_avg'] = total_energy.loc[:,['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']].mean(axis=1)
total_energy['last_ten_avg_ratio'] = total_energy['last_ten_avg']/total_energy['last_ten_avg'].sum()


total_energy

,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,last_three_avg,last_three_avg_ratio,last_five_avg,last_five_avg_ratio,last_ten_avg,last_ten_avg_ratio
geography,,,,,,,,,,,,,,,,,,,,
01,48443104.0,52474784.0,50501051.0,NaN,NaN,46191687.0,47125064.0,48328058.0,47894160.0,48105292.0,44615373.0,42814239.0,46931453.0,44381062.0,44708918.0,0.117134,45369483.8,0.117773,46265154.222222,0.119272
03,11828256.0,NaN,NaN,NaN,NaN,NaN,NaN,12879731.0,NaN,12142401.0,12040139.0,11360176.0,11988910.0,NaN,11674543.0,0.030586,11882906.5,0.030846,12082271.4,0.031148
04,10820806.0,14814646.0,14180538.0,NaN,NaN,12631848.0,14109789.0,NaN,13143009.0,12886987.0,13856826.0,NaN,13658511.0,13142894.0,13400702.5,0.035109,13386304.5,0.034749,13347123.428571,0.034409
05,17053986.0,18575860.0,17759277.0,NaN,16981922.0,17056322.0,NaN,17845930.0,NaN,17529187.0,17899962.0,NaN,17486160.0,NaN,17486160.0,0.045812,17638436.333333,0.045787,17466580.5,0.045029
06,NaN,12257857.0,11590436.0,11597513.0,11505324.0,NaN,11256865.0,11768100.0,11907589.0,11811440.0,12412753.0,11839948.0,NaN,NaN,11839948.0,0.03102,12021380.333333,0.031206,11786002.714286,0.030384
07,6846933.0,7336340.0,6604596.0,6293819.0,6157216.0,5947005.0,5570358.0,5820530.0,5816436.0,NaN,5572359.0,5085944.0,5540824.0,NaN,5313384.0,0.013921,5399709.0,0.014017,5688834.0,0.014666
08,13003118.0,13865234.0,12618370.0,NaN,13293578.0,NaN,NaN,13192723.0,13025898.0,12450362.0,11922463.0,11125217.0,11468596.0,11159723.0,11251178.666667,0.029477,11625272.2,0.030177,12204820.0,0.031464
09,4362266.0,3911532.0,3702660.0,3945523.0,4234474.0,4253226.0,4524762.0,3804524.0,NaN,4035538.0,4063017.0,3649917.0,3930542.0,3716847.0,3765768.666667,0.009866,3879172.2,0.01007,4023649.666667,0.010373
10,NaN,7190890.0,NaN,7351151.0,7560787.0,6995165.0,NaN,7432143.0,7047901.0,NaN,7351238.0,7181648.0,8183041.0,7736893.0,7700527.333333,0.020175,7613205.0,0.019763,7436102.0,0.01917


In [10]:
county_energy_split = total_energy['last_ten_avg_ratio']
county_energy_split.rename('factor', inplace=True)
county_energy_split.to_csv('county_energy_split.csv')
